In [ ]:

import struct
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

def lire_pcx(chemin):
    with open(chemin, "rb") as f:
        data = f.read()

    if len(data) < 128:
        raise ValueError("Fichier PCX invalide")

    # 1. Lecture dynamique du Header
    header = data[:128]

    fabricant = header[0]
    version = header[1]
    compression = header[2]
    bits_pixel = header[3]

    xmin, ymin, xmax, ymax = struct.unpack(
        "<4H", header[4:12]
    )

    largeur = xmax - xmin + 1
    hauteur = ymax - ymin + 1

    dpi_x, dpi_y = struct.unpack("<2H", header[12:16])
    plans = header[65]
    octets_ligne = struct.unpack("<H", header[66:68])[0]

    print("===== INFORMATIONS PCX =====")
    print("Fabricant :", hex(fabricant))
    print("Version :", version)
    print("Compression :", compression)
    print("Bits/pixel/plan :", bits_pixel)
    print("Dimensions :", largeur, "x", hauteur)
    print("Résolution :", dpi_x, "x", dpi_y)
    print("Plans :", plans)
    print("Octets par ligne :", octets_ligne)
    print("Taille fichier :", len(data), "octets")

    if fabricant != 10:
        raise ValueError("Ce fichier n'est pas un PCX valide")

    # Ce décodeur traite les PCX 256 couleurs
    if bits_pixel != 8 or plans != 1:
        raise ValueError(
            "Format non pris en charge : "
            "8 bits et 1 plan requis"
        )

    # 2. Détection automatique de la palette
    if len(data) >= 897 and data[-769] == 12:
        palette = data[-768:]
        raster = data[128:-769]
    else:
        raise ValueError("Palette 256 couleurs introuvable")

    # 3. Décompression dynamique RLE
    pixels = bytearray()
    taille_attendue = octets_ligne * hauteur

    i = 0
    while i < len(raster) and len(pixels) < taille_attendue:
        valeur = raster[i]
        i += 1

        if compression == 1 and valeur >= 192:
            repetitions = valeur & 0x3F

            if i >= len(raster):
                raise ValueError("Données RLE incomplètes")

            couleur = raster[i]
            i += 1
            pixels.extend([couleur] * repetitions)
        else:
            pixels.append(valeur)

    if len(pixels) != taille_attendue:
        raise ValueError("Taille des pixels incorrecte")

    # 4. Reconstruction automatique
    matrice = np.frombuffer(
        bytes(pixels), dtype=np.uint8
    ).reshape(hauteur, octets_ligne)

    # Suppression des octets de remplissage
    matrice = matrice[:, :largeur]

    image = Image.fromarray(matrice, mode="P")
    image.putpalette(palette)

    print("Pixels décodés :", largeur * hauteur)
    print("Décompression terminée !")

    # 5. Affichage
    plt.figure(figsize=(10, 6))
    plt.imshow(image.convert("RGB"))
    plt.title(f"Image PCX : {Path(chemin).name}")
    plt.axis("off")
    plt.show()

    return image


# ===== UTILISATION =====
chemin = input("Entrer le chemin du fichier PCX : ").strip().strip('"')

image = lire_pcx(chemin)

# Sauvegarde optionnelle
reponse = input("Convertir en PNG ? (o/n) : ")

if reponse.lower() == "o":
    sortie = str(Path(chemin).with_suffix(".png"))
    image.save(sortie)
    print("Image sauvegardée :", sortie)
